# 4. 予測する仕組みを作る

下ごしらえの済んだデータを使って、**乗客の情報から生死を言い当てる仕組み**を作る。

この仕組みのことを **モデル** と呼ぶ。
学習用データの445人を見せて法則を覚えさせ、それを評価用データの446人に当てはめる。

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

# 03 で保存した加工後データ。無ければ 03 を実行して作る
train = pd.read_csv("../data/processed/train_processed.csv", index_col=0)
test = pd.read_csv("../data/processed/test_processed.csv", index_col=0)

print("train:", train.shape, " test:", test.shape)

## 1. 答えと手がかりに分ける

下ごしらえを終えた `train` は項目が11個ある。
このうち `survived` だけが**答え**で、残りの10個が**手がかり**になる。

| id | survived | pclass | age | sex_female | ... |
| --- | --- | --- | --- | --- | --- |
| 3 | 1 | 1 | 35.0 | True | ... |
| 4 | 0 | 3 | 35.0 | False | ... |

モデルに覚えさせるときは、この2つを**別々に渡す**。

```
        train（11 個）
  ┌──────────┬──────────────────────────┐
  │ survived │ pclass  age  sex_female  │
  │          │ sibsp   parch  fare  ... │
  └──────────┴──────────────────────────┘
       │                   │
       ▼                   ▼
       y                   X
    答え（1 個）      手がかり（10 個）
```

問題集にたとえると、`X` が問題文で、`y` が巻末の解答。
両方を並べて見せることで、「3等客室の男性はたいてい 0」「女性はたいてい 1」
といった対応を見つけさせる。

答えを混ぜたまま渡してはいけない理由も、このたとえで分かる。
問題文の中に答えが書いてあったら、考えずにそれを写すだけになってしまう。

### なぜ `X` と `y` なのか

名前は何でもよいが、この分野ではこの2文字がほぼ決まりになっている。

- `X` が**大文字**なのは、項目がたくさん並んだ表だから
- `y` が**小文字**なのは、1列しかないから

他の人が書いたコードでも同じ名前で出てくるので、そういうものとして覚えてよい。

### `drop` で項目を落とす

`drop` は行にも列にも使えるので、**どちらを落とすのか**を伝える必要がある。

```python
train.drop(columns=["survived"])   # 項目（列）を落とす
train.drop(index=[3, 4])           # 人（行）を落とす
```

`axis=1`（列）、`axis=0`（行）と番号で書く方法もあり、他の人のコードではよく見かける。
結果は同じなので、ここでは読んで意味が分かる `columns=` を使う。

In [ ]:
y = train["survived"]  # 目的変数
X = train.drop(columns=["survived"])  # 説明変数（survived を除いた残り全部）

print("X:", X.shape, " y:", y.shape)
print("X の列:", list(X.columns))

# test には元から survived が無いので、X と同じ 10 列になっている
print("test と列の並びが一致:", list(X.columns) == list(test.columns))

# 並び順の一致を確認しているのは、モデルが列名を見ていないため。
# 「左から 2 番目は年齢」と位置で覚えるので、test だけ順序が違うと
# 年齢のつもりで運賃を読んでしまい、気づかないまま間違った予測になる

## 2. 採点用のデータを取り分ける

445人を全部覚えさせてしまうと、**できの良さを測る方法がなくなる**。

覚えさせたデータで採点しても、丸暗記しただけのモデルが満点を取れてしまい、
知らない人に通用するのかが分からない。

そこで445人を2つに分ける。

- **学習用**（8割）— モデルに見せる
- **検証用**（2割）— 見せずに隠しておき、採点だけに使う

### `train` という言葉が2か所で出てくる

紛らわしいので、全体の関係を整理しておく。

```
891 人（元の名簿）
├── train.csv  445 人  ← 答えがある
│   ├── X_train  356 人  ← モデルに見せる
│   └── X_valid   89 人  ← 隠しておいて採点に使う
└── test.csv   446 人  ← 答えが無い。コンペ側が持っている
```

`train`（445人）を自分でさらに 4:1 に割ったのが `X_train` と `X_valid`。

`valid` は **validation**（検証）の略。なぜ `test` と呼ばないかというと、
その名前がすでに使われているから。本来この役目は3つに分かれる。

| 名前 | 役目 | 今回 |
| --- | --- | --- |
| train | 見せて覚えさせる | `X_train` |
| validation | 手元で採点して、工夫が効いたか確かめる | `X_valid` |
| test | 最後の本番 | `test.csv`（答えは手元にない） |

`X_valid` の点数は何回測ってもよい。提出する前の自主検査に使う物差し。

In [ ]:
# train_test_split はデータをランダムに2つへ分ける
#   test_size=0.2  : 2割を検証用に取り分ける
#   random_state=0 : 乱数の種を固定する。付けないと実行のたびに分け方が変わり、
#                    精度が上下したとき改良の成果か偶然かを区別できなくなる
#   stratify=y     : 分けた後も生存者の割合が元と同じになるようにする。
#                    偏ると採点の条件が変わってしまうため
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

print("学習用:", X_train.shape, " 検証用:", X_valid.shape)

## 3. モデルを作る

**ロジスティック回帰**という仕組みを使う。名前は難しいが、やっていることは単純。

> 項目ごとに「重み」を決めて、値 × 重みを全部足す。
> その合計を 0〜1 の確率に変換する。

「女性なら生存側に大きく傾ける」といった判断を、重みの大小として覚える。

ただし、そのまま渡すと困ったことが起きる。項目ごとに**数字の大きさが違いすぎる**。

| 項目 | 値の範囲 |
| --- | --- |
| `fare` | 0 〜 512 |
| `age` | 0.67 〜 80 |
| `sex_female` | 0 か 1 |

運賃だけ桁が大きいので、計算がうまく進まない。
実際、何もしないと「計算が終わらなかった」という警告が出る。

そこで **標準化** をはさむ。どの項目も**同じくらいの大きさに揃える**変換で、
これをしてから渡す。

### 標準化すると値がどう変わるか

やっていることは、各項目を

> **平均より上か下か、どのくらい離れているか**

という形に言い換える作業。歳や運賃といった単位が消えて、どの項目も同じ物差しに乗る。

- 平均ぴったりの人は **0**
- 平均より上なら **プラス**
- 平均より下なら **マイナス**

**0〜1 に収める変換ではない**ので、マイナスにも 1 超えにもなる。

In [ ]:
# パイプラインの中で起きているのと同じ変換を、目で見るために単体で実行する
#   .fit() で平均とばらつきを覚え、.transform() でその基準に沿って変換する
scaler = StandardScaler().fit(X_train)
after = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)

cols = ["age", "fare", "sex_female"]
print("変換前");  print(X_train[cols].head(3))
print()
print("変換後");  print(after[cols].head(3).round(3))

# 出力の見方（1 人目の id 393）
#   age  23.0    → -0.452  平均 28.89 歳より少し若いので、少しマイナス
#   fare 113.275 → +1.406  平均 35.31 よりかなり高いので、大きくプラス
#   もとは 23 と 113 で 5 倍違ったのに、変換後はどちらも -1 〜 1 あたりに収まった

In [ ]:
# 項目ごとに、変換前後の平均とばらつきを比べる
pd.DataFrame({
    "変換前 平均": X_train[cols].mean(),
    "変換前 ばらつき": X_train[cols].std(ddof=0),
    "変換後 平均": after[cols].mean(),
    "変換後 ばらつき": after[cols].std(ddof=0),
}).round(3)

# 出力の見方
#   変換前は平均が 28.9 / 35.3 / 0.4 とばらばらで、ばらつきも 13 / 55 / 0.5 と桁違い
#   変換後はどの項目も 平均 0・ばらつき 1 にそろう。これが「同じ物差しに乗せる」ということ

In [ ]:
# 変換後、値がどこまで動くのかを見る
after[cols].agg(["min", "max"]).round(2)

# 出力の見方
#   age は -2.17 〜 3.92、fare は -0.64 〜 8.60
#   0 〜 1 には収まらない。fare の 8.60 は、512 を払った人が
#   「平均から飛び抜けて高い」ことを表している

In [ ]:
# make_pipeline は複数の処理を一列に繋げる
#   ここでは 標準化 → ロジスティック回帰 の順に通る
#
# 繋げる理由は、標準化の基準を「学習用データだけ」から決めるため。
# 別々に書くと、検証用や評価用まで混ぜて平均を計算してしまいがちで、
# 本当は知らないはずの情報が紛れ込む
model = make_pipeline(StandardScaler(), LogisticRegression())

# .fit(手がかり, 答え) で覚えさせる。これだけでモデルができる
#   セルの最後に置くと、組み立てた工程がそのまま表示される
model.fit(X_train, y_train)

### `.fit()` は何をしたのか

モデルの正体は、**項目ごとの重みと、全員に一律で足す数字**だけ。
それ以上のものは持っていない。`.fit()` はこの数字を決める作業をしている。

始める前は全部 0 で、誰を入れても答えが 0.5 になる状態。そこから、

1. 今の重みで356人の生存確率を計算する
2. 実際の答えと比べて、どれだけずれているかを測る
3. ずれが小さくなる方向に重みを少し動かす
4. 1 に戻る

を繰り返し、これ以上ずれが小さくならなくなったら終わる。

**356人のデータそのものはモデルに残らない。** 残るのは絞り出された数字だけ。

In [ ]:
# fit が決めた重みを見る。model[-1] は一列に繋げた最後、ロジスティック回帰そのもの
pd.Series(model[-1].coef_[0], index=X.columns).round(3)

# 出力の見方
#   プラスなら生存側、マイナスなら死亡側に傾ける
#   sex_female +0.654 / sex_male -0.654、pclass -0.772 が大きい

In [ ]:
print("全員に足す数字:", round(model[-1].intercept_[0], 3))

# これは項目と関係なく、全員に一律で足される数字。
# 全体としてどちら寄りに判定するかを調整している（下駄をはかせるようなもの）

## 4. 成績を確かめる

`.score()` は**正解率**、つまり何割当たったかを返す。
**見せていない検証用データでの点数**が本当の実力になる。

比べる基準として、02 で見た「全員が助からなかったと答えるだけ」の正解率も並べる。
これを超えていなければ、何も学べていないのと同じ。

In [ ]:
print("検証用データでの正解率:", round(model.score(X_valid, y_valid), 4))
print("学習用データでの正解率:", round(model.score(X_train, y_train), 4))
print("全員「助からない」と答えた場合:", round(1 - y_valid.mean(), 4))

# 出力の見方
#   検証 0.7528 に対して基準 0.5955。約 16 ポイント上回っており、法則を捉えられている
#   学習側が高いのは、係数がその356人に合わせて決められたため。
#   自分に合わせて作った物差しで自分を測っているようなもの

### 学習用と検証用の違い

どちらも**同じ重みを使った同じ計算**で、違うのは誰に当てはめたかだけ。

| | 人数 | モデルは見たか |
| --- | --- | --- |
| 学習用 | 356 | 見た。答えも一緒に |
| 検証用 | 89 | **見ていない** |

問題集にたとえると、学習用は答え合わせ済みの例題、検証用は初めて解く模試。
**実力を表すのは検証用の方**で、提出したときの点数もこれが目安になる。

ただし**1回の数字を細かく読みすぎないこと。** 検証用は89人しかいない。

```
1 人の当たり外れ = 1.12 ポイント
```

数ポイントの差は、数人分の違いにすぎない。実際、分け方を変えて試すと
検証用の点数は 0.72〜0.89 の範囲で動き、検証用の方が高く出る回もある。

差が**大きく開いたとき**だけ、丸暗記のサインとして読む。
たとえば決定木（条件分岐を積み重ねる別の仕組み）を制限なしで使うと、
学習用 0.99 / 検証用 0.72 まで開く。覚え込みすぎて、初見に通用しなくなった状態。

### 答えを知っているのに、なぜ学習用でも 100% にならないのか

**予測するとき、モデルに答えは渡っていないから。**

```
覚えるとき: 手がかり 10 個 + 答え  →  重みを決める
予測するとき: 手がかり 10 個だけ   →  重みに通して確率を出す
```

そしてモデルが持てるのは重みと一律の数字だけで、
**356人分の答えをしまっておく場所がない**。
できるのは「こういう人はだいたいこう」という**法則の形に圧縮すること**だけ。

だから例外は必ず外す。

- 3等の男性は 175 人中 151 人（86.3%）が亡くなっている。
  法則としては「死亡」に賭けるのが正しく、それでも助かった 24 人は外れる
- 1等の女性は 53 人中 94.3% が助かっている。
  ほぼ確実に助かる条件なので、亡くなった人は外れる

もうひとつの理由は、この仕組みが**単純**なこと。
できるのは項目ごとに重みを掛けて足すだけで、
「3等でも女性で子連れなら助かりやすい」といった**条件の組み合わせ**は表現できない。

この単純さは弱点でもあるが、覚え込みすぎない強さでもある。

## 5. 全員分で覚え直して予測する

成績の見当が付いたので、今度は**445人全員**を使って覚え直す。

採点用に取り分けていた2割も、本番では使わない手はない。
覚えさせるデータは多いほどよく、成績の確認はもう済んでいる。

In [ ]:
# 同じ構成のモデルを、今度は train 全体で学習する
model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X, y)

In [ ]:
# .predict_proba() は確率を返す。列が2つあることに注意
proba = model.predict_proba(test)

print("形:", proba.shape)
print("先頭1人の値:", proba[0].round(4), " 合計:", proba[0].sum())

# 出力の見方
#   左の列 = 助からない確率、右の列 = 生き残る確率。2つ足すと必ず 1 になる
#   提出に必要なのは生存確率なので、右の列だけを取り出す

In [ ]:
# [:, 1] は「全ての行の、1番目の列」という指定（0 から数えるので 1 が右の列）
pred = model.predict_proba(test)[:, 1]

print("先頭5人の生存確率:", pred[:5].round(6))
print("生存確率が 0.5 を超えた人数:", (pred > 0.5).sum(), "/", len(pred))

## 6. どの項目が効いたか

覚えた重みを見ると、モデルが何を根拠に判断しているかが分かる。
標準化で同じ物差しに乗せてあるので、項目どうしで大きさを比べられる。

プラスなら生存側、マイナスなら死亡側に傾ける。

In [ ]:
# model[-1] は一列に繋げた最後、ロジスティック回帰そのもの
coef = pd.Series(model[-1].coef_[0], index=X.columns).sort_values()
coef

# 出力の見方
#   pclass -0.797  : 等級の数字が大きい（下の客室）ほど死亡側。最も強い
#   sex_male -0.637 / sex_female +0.637 : 男性は死亡側、女性は生存側
#   age -0.445     : 年齢が高いほど死亡側
#
#   02 の相関係数では -0.081 と弱く見えた age が、ここでは 3 番目に効いている。
#   相関係数は項目を 1 つずつ単独で見るのに対し、
#   モデルは他の項目の影響を差し引いてから age の効き方を測るため

### 1人分を手で追ってみる

`predict_proba` の中で起きていることを、実際にばらして見る。
やっているのは「標準化した値 × 重み」を全部足して、確率に変換するだけ。

In [ ]:
import numpy as np

# 評価用データの1人目を取り出す
person = test.iloc[[0]]
scaler, clf = model[0], model[1]

z = scaler.transform(person)[0]  # 標準化した値
contrib = pd.Series(z * clf.coef_[0], index=X.columns)  # 値 × 係数

print("寄与が大きい順:")
print(contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(5).round(3))
print()
total = contrib.sum() + clf.intercept_[0]
print("合計 + 切片 =", round(total, 3))
print("確率に変換  =", round(1 / (1 + np.exp(-total)), 6))
print("predict_proba と一致:", np.isclose(1/(1+np.exp(-total)), model.predict_proba(person)[0, 1]))

# 出力の見方
#   マイナスの寄与が積み上がると合計が下がり、確率も下がる
#   最後の変換で、どんな合計値でも必ず 0〜1 に収まる
#   これを 446 人分繰り返したものが pred

## 7. 予測結果を保存する

05 で提出用のファイルに整えるため、id と紐付けて書き出す。

書き出す `pred_test.csv` は、**446人分の生存確率**を id と並べただけのファイル。

```
id,pred
0,0.101915     ← id 0 の人は 10.2% の確率で生還
1,0.931212
```

まだ提出できる形ではない。提出用はヘッダ行を付けない決まりになっている。
確率の値はそのまま使うので、整形は 05 で行う。

`id` を一緒に書き出しているのが要点で、並び順だけに頼ると、
どこかで行が入れ替わったときに**別人の予測を提出してしまう**。

In [ ]:
# test.index が乗客の id。並び順に依存せず対応が取れるよう、id を付けて保存する
pd.Series(pred, index=test.index, name="pred").to_csv(
    "../data/processed/pred_test.csv", index=True
)

print("保存した")

## この章のまとめ

- `survived` を答え `y`、残り10個を手がかり `X` として分けた
- 445人のうち2割を採点用に取り分けて覚えさせ、**検証用で 75.3%** の正解率。
  基準の 59.6%（全員「助からない」と答えた場合）を約16ポイント上回った
- 項目ごとに数字の大きさが違いすぎるので、標準化をはさんだ。
  入れないと計算が終わらず警告が出る
- 成績を確かめた後、445人全員で覚え直してから予測した
- 効いている順は 客室の等級・性別・年齢。
  単独では弱く見えた年齢が、他の項目の影響を除くと3番目に効いている
- 446人分の生存確率を `data/processed/pred_test.csv` に保存した